# Αποστολή 8: 📦 Αυτόνομος Πράκτορας Διάγνωσης (Diagnostics Agent)

Καλώς όρισες στην Αποστολή 8!

Εδώ θα κατασκευάσεις έναν πραγματικό **Αυτόνομο Πράκτορα (Autonomous Agent)**! 
Ο πράκτοράς σου θα συνδυάζει **RAG (Retrieval-Augmented Generation)** και **Tool Calling**:
1. Θα διαβάζει ένα τοπικό αρχείο καταγραφής (diagnostics log).
2. Θα στέλνει τα δεδομένα στη Gemma για να εντοπίσει αν υπάρχει διαρροή οξυγόνου.
3. Αν η Gemma εντοπίσει διαρροή, η Python θα ενεργοποιεί αυτόματα το κατάλληλο εργαλείο επισκευής (`seal_leak`)!

### 🎯 Μάθημα 45: Διάβασμα Αρχείου Καταγραφής (RAG Setup)

**Η Ιστορία:** Πρέπει να διαβάσεις το αρχείο `diagnostics.txt` που περιέχει την κατάσταση των συστημάτων οξυγόνου του σταθμού.

**Στόχος:** Δημιούργησε ένα αρχείο `diagnostics.txt` με περιεχόμενο `'Sector 4: Oxygen Level 12% [WARNING: PRESSURE LEAK DETECTED] [VALVE_ID: O2-LEAK-4]'` χρησιμοποιώντας την Python, διάβασέ το και αποθήκευσε το περιεχόμενο στη μεταβλητή `log_data`.

In [ ]:
# 💻 Γράψε το diagnostics.txt, διάβασέ το και αποθήκευσε στη log_data:
with open('diagnostics.txt', 'w') as f:
    f.write('Sector 4: Oxygen Level 12% [WARNING: PRESSURE LEAK DETECTED] [VALVE_ID: O2-LEAK-4]')

with open('diagnostics.txt', 'r') as f:
    log_data = f.read()
print(log_data)

# === AUTOMATED CHECK ===
try:
    assert 'log_data' in locals() or 'log_data' in globals(), "Δεν δημιούργησες τη μεταβλητή 'log_data'!"
    assert "O2-LEAK-4" in log_data, "Το αρχείο καταγραφής δεν περιέχει τα σωστά δεδομένα!"
    print("\n🎉 [SYSTEM] Great job! Προετοίμασες τα δεδομένα RAG με επιτυχία!")
    print("👉 Type 'progress' to the Pi Agent για να δει τι κατάφερες!")
    
    try:
        import sys
        sys.path.append("/opt/space-station-academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 45: Diagnostics Log Setup")
    except Exception:
        pass
except AssertionError as e:
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 46: Ανάλυση Δεδομένων με την AI (Context Analysis)

**Η Ιστορία:** Τώρα, θα στείλεις τα δεδομένα του αρχείου στη Gemma και θα τη ρωτήσεις αν ανιχνεύει διαρροή οξυγόνου.

**Στόχος:** Στείλε στη Gemma ως System prompt: `'You are a station safety officer. If you detect a leak, reply ONLY with the text "LEAK_FOUND:" followed by the VALVE_ID. Otherwise, reply "SAFE".'` και ως user prompt τα δεδομένα της μεταβλητής `log_data`. Αποθήκευσε την απάντηση στη μεταβλητή `analysis_result`.

In [ ]:
import ollama

# 💻 Στείλε τα δεδομένα RAG στη Gemma για ανάλυση και αποθήκευσε στη analysis_result:
analysis_result = ollama.chat(
    model='gemma4:e4b',
    messages=[
        {
            'role': 'system',
            'content': 'You are a station safety officer. If you detect a leak, reply ONLY with the text "LEAK_FOUND:" followed by the VALVE_ID. Otherwise, reply "SAFE".'
        },
        {
            'role': 'user',
            'content': log_data
        }
    ]
)
analysis_text = analysis_result['message']['content'].strip()
print("AI Analysis:", analysis_text)

# === AUTOMATED CHECK ===
try:
    assert 'analysis_result' in locals() or 'analysis_result' in globals(), "Δεν δημιούργησες τη μεταβλητή 'analysis_result'!"
    assert "LEAK_FOUND" in analysis_text, "Η AI απέτυχε να εντοπίσει τη διαρροή!"
    print("\n🎉 [SYSTEM] Great job! Η AI ανέλυσε επιτυχώς τα δεδομένα RAG!")
    print("👉 Type 'progress' to the Pi Agent για να δει τι κατάφερες!")
    
    try:
        import sys
        sys.path.append("/opt/space-station-academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 46: RAG Analysis")
    except Exception:
        pass
except AssertionError as e:
    print(f"\n❌ [SYSTEM] Error: {e}")

### 🎯 Μάθημα 47: Αυτόματο Tool Calling (Autonomous Repair)

**Η Ιστορία:** Ο πράκτορας πρέπει τώρα να πάρει την απόφαση! Αν η απάντηση της AI περιέχει τη λέξη `'LEAK_FOUND'`, ο πράκτορας θα καλέσει τη συνάρτηση `seal_leak(valve_id)` με το κατάλληλο ID.

**Στόχος:** Ορίστε τη συνάρτηση `seal_leak(valve_id)` που επιστρέφει `True` και εμφανίζει `'Sealing leak on valve...'`. Αναλύστε την `analysis_text` και αν περιέχει `'LEAK_FOUND'`, απομονώστε το VALVE_ID (μετά την άνω-κάτω τελεία `:`) και καλέστε τη συνάρτηση, αποθηκεύοντας το αποτέλεσμα στη μεταβλητή `repair_status`.

In [ ]:
def seal_leak(valve_id):
    print(f"[TOOL] Activating remote welding micro-drone... Sealing leak on valve: {valve_id}!")
    return True

# 💻 Ανίχνευσε αν βρέθηκε διαρροή και κάλεσε τη seal_leak αυτόνομα:
repair_status = False
if "LEAK_FOUND" in analysis_text:
    valve_id = analysis_text.split(":")[1].strip()
    repair_status = seal_leak(valve_id)

# === AUTOMATED CHECK ===
try:
    assert 'repair_status' in locals() or 'repair_status' in globals(), "Δεν δημιούργησες τη μεταβλητή 'repair_status'!"
    assert repair_status == True, "Ο αυτόνομος πράκτορας απέτυχε να επισκευάσει τη διαρροή!"
    print("\n🎉 [SYSTEM] AUTONOMOUS REPAIR COMPLETE! Ο πράκτοράς σου εντόπισε και επισκεύασε τη διαρροή αυτόνομα!")
    print("👉 Κάνε κλικ στο ΕΠΟΜΕΝΟ ΜΑΘΗΜΑ (NEXT) για να συνεχίσεις στην Αποστολή 9!")
    
    try:
        import sys
        sys.path.append("/opt/space-station-academy/.pi")
        import tracker
        tracker.update_progress("Mission 8, Μάθημα 47: Autonomous Tool Calling")
    except Exception:
        pass
except AssertionError as e:
    print(f"\n❌ [SYSTEM] Error: {e}")